In [ ]:
# Cài đặt thư viện của tác giả để nó biết cách "đọc" file của bạn
!pip install vieneu

from google.colab import drive
drive.mount('/content/drive')


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 62.9 MB/s eta 0:00:00
  Created wheel for perth: filename=perth-1.0.0-py3-none-any.whl size=1886 sha256=92461f6b506b610dd1593bc45f18e5b48b95db0c8c22999d99ea21a849430c2c
  Stored in directory: /root/.cache/pip/wheels/22/00/da/6a89e8aa931ca478750f4a2e1ce854255351816b2a8ca7b6a7
Successfully built perth
Mounted at /content/drive


In [ ]:
# Cell 2 3

In [ ]:
from vieneu import Vieneu
import os

class Module_TTS:
    def __init__(self, model_config, resource_config):
        # Đường dẫn khớp 100% với thư mục Drive của bạn
        self.model_path = model_config["model_path"]

        print(f"[Module TTS] Đang nhờ thư viện chính chủ đọc file từ: {self.model_path}")

        # Ép hệ thống chui vào đúng thư mục của bạn để đọc file (Offline hoàn toàn)
        os.environ["HF_HOME"] = self.model_path

        # Khởi tạo mô hình (Tự động nhận diện chạy trên GPU T4 của Colab)
        self.model = Vieneu()

    def synthesize_stream(self, text_stream, max_chars=150):
        """
        Nhận từng đoạn văn bản từ LLM (Streaming) và cắt thành câu
        """
        buffer = ""
        for chunk in text_stream:
            buffer += chunk

            # Ngắt đoạn để đọc ngay nếu có dấu câu hoặc dài quá giới hạn
            if any(punct in chunk for punct in ['.', ',', '!', '?', '\n']) or len(buffer) >= max_chars:
                if buffer.strip():
                    yield self._generate_audio(buffer.strip())
                buffer = ""

        if buffer.strip():
            yield self._generate_audio(buffer.strip())

    def _generate_audio(self, text):
        """
        Chuyển đoạn văn bản thành âm thanh
        """
        # Dùng hàm của tác giả để sinh ra mảng sóng âm thanh
        audio_array = self.model.infer(text=text)

        # Tần số lấy mẫu mặc định của mô hình VieNeu v3 Turbo là 48kHz
        sampling_rate = 48000

        return audio_array, sampling_rate


In [ ]:
from IPython.display import Audio, display

# 1. Cấu hình hệ thống
model_config = {
    # Đường dẫn trỏ thẳng vào thư mục bạn đã upload trên Drive
    "model_path": "/content/drive/MyDrive/models/vieneu-tts-v3-turbo"
}
resource_config = {
    "device": "cuda", # Cho phép chạy GPU để đảm bảo tốc độ nhanh nhất
    "gpu_layers": 20,
    "cpu_layers": 8
}

# 2. Khởi tạo Module TTS
tts_module = Module_TTS(model_config, resource_config)

# 3. Giả lập luồng văn bản từ Module LLM truyền sang
def mock_llm_stream():
    yield "Xin chào, "
    yield "đây là giọng nói "
    yield "được sinh ra "
    yield "từ mô hình VieNeu "
    yield "đọc trực tiếp từ thư mục trên Google Drive của tôi!"

# 4. Chạy hệ thống và phát tiếng trực tiếp ra loa
print("\n[HỆ THỐNG] Đang nhận chữ và đọc...")

for audio_array, sampling_rate in tts_module.synthesize_stream(mock_llm_stream()):
    # Thay vì lưu file rác, lệnh này sẽ hiện thanh Play Audio phát tiếng ngay lập tức trên Colab
    display(Audio(data=audio_array, rate=sampling_rate, autoplay=True))



[Module TTS] Đang nhờ thư viện chính chủ đọc file từ: /content/drive/MyDrive/models/vieneu-tts-v3-turbo


[transformers] You are using a model of type `vieneu_v3` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Loading weights:   0%|          | 0/374 [00:00<?, ?it/s]


[HỆ THỐNG] Đang nhận chữ và đọc...


In [ ]:
#Mô phỏng quá trình thực hiện code dưới cho phép đổi giọng ( đọc comment )

In [ ]:
from vieneu import Vieneu
import os

class Module_TTS:
    def __init__(self, model_config, resource_config):
        self.model_path = model_config["model_path"]
        print(f"[Module TTS] Khởi tạo hệ thống, đọc file từ: {self.model_path}")

        os.environ["HF_HOME"] = self.model_path
        self.model = Vieneu()

        # Tải sẵn danh sách các giọng mặc định vào bộ nhớ
        self.danh_sach_giong = self.model.list_preset_voices()

    def synthesize_stream(self, text_stream, max_chars=80):
        buffer = ""
        for chunk in text_stream:
            buffer += chunk

            # Logic cân bằng: Chờ dấu chấm, nhưng nếu dài hơn 40 ký tự thì ngắt tạm ở dấu phẩy
            if any(punct in chunk for punct in ['.', '!', '?', '\n']) or \
               (any(punct in chunk for punct in [',']) and len(buffer) > 40) or \
               len(buffer) >= max_chars:

                if buffer.strip():
                    yield self._generate_audio(buffer.strip())
                buffer = ""

        if buffer.strip():
            yield self._generate_audio(buffer.strip())

    def _generate_audio(self, text):
        # =========================================================
        #  HƯỚNG DẪN CHỈNH GIỌNG NÓI Ở ĐÂY
        # Bạn thay đổi con số ở biến `index_giong` dưới đây (0, 1, 2, 3...)
        # =========================================================
        index_giong = 2

        # Lấy ID của cái giọng tương ứng với con số bạn vừa chọn
        giong_chon = self.model.get_preset_voice(self.danh_sach_giong[index_giong][1])

        # Chuyển văn bản thành âm thanh với ĐÚNG GIỌNG ĐÓ
        audio_array = self.model.infer(text=text, voice=giong_chon)

        return audio_array, 48000


In [ ]:
import numpy as np
from IPython.display import Audio, display

model_config = {
    "model_path": "/content/drive/MyDrive/models/vieneu-tts-v3-turbo"
}
resource_config = {
    "device": "cuda",
    "gpu_layers": 20,
    "cpu_layers": 8
}

tts_module = Module_TTS(model_config, resource_config)

def mock_llm_stream():
    long_text = 'Hệ thống RAG, kết hợp cùng mô hình ngôn ngữ lớn LLM, đã đem lại những cải tiến vượt bậc! Bạn có biết điều tuyệt vời nhất là gì không? Thay vì phải chờ đợi hệ thống suy nghĩ xong toàn bộ đoạn văn, luồng dữ liệu streaming của Module TTS sẽ lập tức "đớp" lấy từng cụm từ, và chuyển đổi thành âm thanh ngay khi gặp dấu phẩy, giúp trải nghiệm của người dùng trở nên mượt mà, tự nhiên, và không hề có độ trễ!'

    for word in long_text.split():
        yield word + " "

print("\n[HỆ THỐNG] Đang nhận chữ và đọc...")
tat_ca_am_thanh = []
tan_so = 48000
# Hệ thống vẫn sinh ra từng mảnh nhỏ (Streaming)
for audio_array, sampling_rate in tts_module.synthesize_stream(mock_llm_stream()):
    tat_ca_am_thanh.append(audio_array) # Bỏ mảnh âm thanh vào túi
    tan_so = sampling_rate

# Lấy keo dán (concatenate) tất cả mảnh nhỏ trong túi thành 1 mảng dài liền mạch
am_thanh_lien_mach = np.concatenate(tat_ca_am_thanh)

# Hiển thị ĐÚNG 1 thanh Audio duy nhất, đọc một lèo từ đầu đến cuối!
display(Audio(data=am_thanh_lien_mach, rate=tan_so, autoplay=True))


[Module TTS] Khởi tạo hệ thống, đọc file từ: /content/drive/MyDrive/models/vieneu-tts-v3-turbo


[transformers] You are using a model of type `vieneu_v3` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Loading weights:   0%|          | 0/374 [00:00<?, ?it/s]


[HỆ THỐNG] Đang nhận chữ và đọc...
